<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/nlp-labs/blob/main/Session4/1-Text-classification-BERT.ipynb)


# Maestría en Inteligencia Artificial  
## Procesamiento de Lenguaje natural
### Sesión 3 - Práctica

---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

# Introducción

# Clasificación de textos - Análisis de reseñas y/u opiniones

En este notebook se aborda el problema de clasificación de texto aplicado al análisis de reseñas y opiniones en lenguaje natural. El objetivo principal es identificar automáticamente el sentimiento  presente en un conjunto de descripciones textuales utilizando técnicas de representación semántica.

# 1. Configurar entorno

En esta sección se configuran las librerías y dependencias necesarias para el análisis de datos y procesamiento de lenguaje natural. Esto garantiza que el entorno esté listo para cargar, limpiar y analizar las reseñas.

In [ ]:
import importlib.metadata
import warnings

warnings.filterwarnings('ignore')

installed_packages = [dist.metadata['Name'].lower() for dist in importlib.metadata.distributions()]
IN_COLAB = 'google-colab' in installed_packages

In [ ]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
# !test '{IN_COLAB}' = 'True' && sudo apt-get update -y  > /dev/null 2>&1
# !test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y > /dev/null 2>&1
# !test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2 > /dev/null 2>&1
# !test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1 > /dev/null 2>&1
# !test '{IN_COLAB}' = 'True' && pip install lightning datasets 'transformers[torch]'==4.41.2 sentence-transformers > /dev/null 2>&1
!test '{IN_COLAB}' = 'True' && !pip install torchinfo evaluate > /dev/null 2>&1
# !test '{IN_COLAB}' = 'True' && !pip install torchinfo > /dev/null 2>&1

# 2. Recopilación de datos

Para el presente análisis se utilizará el conjunto de datos mteb/SpanishSentimentClassification, disponible en Hugging Face, el cual contiene reseñas y opiniones en español relacionadas con servicios de hospedaje. Cada registro del dataset se encuentra etiquetado según su polaridad de sentimiento, clasificándose en categorías positivas o negativas.

Este conjunto de datos permitirá evaluar modelos de representación semántica y clasificación automática en una tarea binaria de análisis de sentimiento aplicada a opiniones reales de usuarios.

In [ ]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('somosnlp/spa_climate_detection', split='train')
dataset

In [ ]:
dataset[1]

In [ ]:
for i in range(5):
    print(dataset[i])

In [ ]:
from collections import Counter
Counter(dataset["answer"])

In [ ]:
import numpy as np
np.mean([len(t.split()) for t in dataset["question"]])

In [ ]:
text_lengths = [len(row['question']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

# 3. Analisis Exploratorio



## Estructura del Dataset

In [ ]:
df = dataset.to_pandas()
df.head()

Revisamos los primero 5 registros del dataset para validar estructura y valores

## Análisis de Distribución

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Extraer etiquetas
labels = dataset["answer"]

# Contar clases
counter = Counter(labels)

# Mostrar conteo
print("Distribución absoluta:")
print(counter)

# Calcular proporciones
total = sum(counter.values())
print("\nDistribución porcentual:")
for cls, freq in counter.items():
    print(f"Clase {cls}: {freq} ({freq/total:.2%})")

# Graficar
plt.figure()
plt.bar(counter.keys(), counter.values())
plt.xticks(list(counter.keys()))
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.title("Distribución de clases")
plt.show()

Podemos observar que se presenta un desbalanceo de clases con un 82.2% de los comentarios positivos en el dataset. El resto pertenecen a comentarios negativos


## Análisis del Texto

In [ ]:
import numpy as np

text_lengths = [len(row['question'].split()) for row in dataset]

print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"{p}%: {np.percentile(text_lengths, p)}")

El promedio de palabras en cada texto es de 16 palabras lo que sugiere textos moderadamente cortos, tipicos en sitios de reviews

In [ ]:
df['Palabras por clase'] = df['question'].str.split().apply(len)
df.groupby('answer')['Palabras por clase'].median()

In [ ]:
df.boxplot('Palabras por clase', by='answer', grid=False, showfliers=False, color='black')
plt.suptitle('')
plt.xlabel('')
plt.show()

## Vocabulario

In [ ]:
from collections import Counter

all_words = []
for text in dataset["question"]:
    all_words.extend(text.split())

vocab = Counter(all_words)
print("Tamaño del vocabulario:", len(vocab))

El dataset cuenta con 3041 palabras unicas lo que indica un vocabulario moderado y una complejidad lexica relativamente baja

## Análisis de Ruido

In [ ]:
df.sample(5)

Podemos ver aleatoriamente algunos registros para rectificar la clasificacion o valor de la etiqueta vs el texto

## Palabras más frecuentes y N-Gramas por clase

Analizamos las palabras y bigramas más frecuentes en cada polaridad, excluyendo stopwords, para identificar el vocabulario diferenciador entre comentarios positivos y negativos.

In [ ]:
from collections import Counter
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("spanish"))
extra_stopwords = {
    'de', 'la', 'el', 'en', 'y', 'a', 'los', 'las', 'del', 'se', '(', ')',
    'que', 'con', 'un', 'una', 'es', 'no', 'lo', 'su', 'por', 'al', ',', '.', 'más', 'le', 'me', 'mi'
}
stop_words.update(extra_stopwords)

def top_words_by_class(ds, label, n=15, sw=None):
    words = []
    for row in ds:
        if row['answer'] == label:
            words.extend(row['question'].lower().split())
    if sw:
        words = [w for w in words if w not in sw]
    return Counter(words).most_common(n)

def get_bigrams(text):
    tokens = text.lower().split()
    return [f"{tokens[i]} {tokens[i+1]}" for i in range(len(tokens) - 1)]

top_pos = top_words_by_class(dataset, 1, sw=stop_words)
top_neg = top_words_by_class(dataset, 0, sw=stop_words)

bg_pos, bg_neg = Counter(), Counter()
for row in dataset:
    bgs = get_bigrams(row['question'])
    if row['answer'] == 1:
        bg_pos.update(bgs)
    else:
        bg_neg.update(bgs)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, top, title, color in [
    (axes[0, 0], top_pos,                'Top 15 palabras — Relacionado', 'steelblue'),
    (axes[0, 1], top_neg,                'Top 15 palabras — No relacionado',  'salmon'),
    (axes[1, 0], bg_pos.most_common(10), 'Top 10 bigramas — Relacionado', 'steelblue'),
    (axes[1, 1], bg_neg.most_common(10), 'Top 10 bigramas — No relacionado',  'salmon'),
]:
    terms, freqs = zip(*top)
    ax.barh(list(terms)[::-1], list(freqs)[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Frecuencia')

plt.tight_layout()
plt.show()

- **Hay un fuerte efecto del desbalance de clases**: en la clase positiva aparecen frecuencias mucho más altas que en la negativa, lo cual es consistente con que el dataset tiene más ejemplos positivos.

- **Palabras compartidas entre clases** (como *hotel*, *habitaciones*, *personal*): indican vocabulario común del dominio (reseñas de hospedaje), pero **no necesariamente discriminativo** por sí solo.

- **Clase positiva**: términos como *muy*, *bien*, *buena*, *buen*, *excelente* aparecen con más peso y reflejan lenguaje de satisfacción.

- **Clase negativa**: aparecen con más relevancia términos asociados a queja (por ejemplo, *peor*, *mal*, *cama*, *limpieza* en ciertos contextos), aunque con menor volumen total.

## Sobre los n-gramas (bigramas)

- Los bigramas más frecuentes (ej. *de la*, *en la*, *lo que*) son mayormente **estructuras funcionales del español**, útiles para contexto pero poco discriminativas.

- Para separar mejor polaridad, conviene identificar bigramas más semánticos, por ejemplo combinaciones tipo:
    - positivo: *muy bueno*, *excelente servicio*, *bien ubicado*
    - negativo: *muy malo*, *no volvería*, *pésimo servicio*


# 4. Tokenizador

En esta etapa se realiza la construcción y entrenamiento del tokenizador a partir del corpus de texto disponible. Para ello, se utiliza el tokenizador base de GPT-2 mediante la librería Hugging Face Transformers, el cual permite aprender una representación eficiente del lenguaje a partir de los datos.

El proceso consiste en recorrer el dataset por lotes y entrenar un nuevo tokenizador que genera un vocabulario de hasta 50.000 tokens, utilizando como base el alfabeto de bytes convertido a caracteres Unicode. Esto permite que el modelo pueda representar cualquier carácter del texto y manejar adecuadamente palabras desconocidas mediante sub-tokens.

Como resultado, el texto es transformado en secuencias de tokens numéricos, lo que permite que posteriormente los modelos de aprendizaje automático puedan procesar la información textual de manera estructurada y eficiente.

In [ ]:
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from tokenizers.pre_tokenizers import ByteLevel

model_name = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
tokenizer.pad_token = '[PAD]'
tokenizer("hola mundo!!", max_length=10, truncation=True, padding='max_length').tokens()

Revisamos el tokenizador obtenido

In [ ]:
tokenizer.vocab_size

In [ ]:
tokenizer.model_max_length

In [ ]:
tokenizer.model_input_names

In [ ]:
tokens = sorted(tokenizer.vocab.items(), key=lambda x: x[1], reverse=False)
print(f"Vocabulario: {tokenizer.vocab_size} tokens")
print("Primeros 15 tokens:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[:15]])
print("15 tokens de en medio:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[20000:20020]])
print("Últimos 15 tokens:")
print([f"{tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[-15:]])

# 5. Definición del Dataset de PyTorch

Ahora definimos una clase Dataset de PyTorch para manejar los datos del problema de clasificación de sentimientos. Esta clase se encargará de:

- Preparar y procesar los textos de entrada.

- Aplicar el tokenizador para convertir el texto en representaciones numéricas.

- Organizar las etiquetas de sentimiento correspondientes.

- Retornar los datos en formato de tensores listos para el entrenamiento del modelo.

In [ ]:
!pip install torchinfo

In [ ]:
id2category = dict(enumerate(np.unique(df['answer'])))
category2id = {v: k for k, v in id2category.items()}

In [ ]:
import torch
from torchinfo import summary
from transformers import AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer("hola mundo!!!", max_length=10, truncation=True, padding='max_length', return_tensors='pt')

print(f"Input Shapes & Types:")
print({k: (v.shape, v.dtype) for k, v in inputs.items()})

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)

# Congelamos los pesos del modelo base para usarlo como featurizer solamente.
for param in model.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

In [ ]:
modules = [m for m, _ in model.named_modules()]
modules

In [ ]:
with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
print({k: v.shape for k, v in outputs.items()})

In [ ]:
outputs

In [ ]:
model.classifier

### Instanciamos el Dataset y creamos los DataLoaders

Dividimos el dataset en conjuntos de entrenamiento (80%), validación (10%) y prueba (10%).

In [ ]:
training_dataset = dataset.train_test_split(train_size=0.8)
validation_dataset = training_dataset['test'].train_test_split(train_size=0.5)

In [ ]:
from datasets.dataset_dict import DatasetDict

new_dataset = DatasetDict({
    'train': training_dataset['train'],
    'val': validation_dataset['train'],
    'test': validation_dataset['test'],
})
new_dataset

In [ ]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer(examples['question'], max_length=max_len, truncation=True, padding='max_length')
    return _preprocess_function

def tokenize(max_len: int = 8):
    def _tokenize(batch):
        return tokenizer(batch['question'], max_length=max_len, truncation=True, padding='max_length')
    return _tokenize

def category_names_2_ids(batch):
    batch['label'] = category2id[batch['answer']]
    return batch

In [ ]:
tokenized_dataset = new_dataset.map(preprocess_function(max_len=512), batched=True)
tokenized_dataset = tokenized_dataset.map(category_names_2_ids)
tokenized_dataset

In [ ]:
!pip install evaluate

In [ ]:
from transformers import Trainer, TrainingArguments
from typing import Dict, Any
import evaluate

# Definimos la función métrica de calidad
accuracy = evaluate.load("accuracy")

def compute_metrics(pred) -> Dict[str, Any]:
    """compute metrics

    Esta función será invocada en
    cada epoca y la utilizaremos para
    calcular la métrica de calidad.
    """
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Retorna un diccionario como {'nombre-metrica': valor}
    acc = accuracy.compute(predictions=preds, references=labels)
    return acc


batch_size = 8 if IN_COLAB else 4
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./hf',
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='tensorboard'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time
trainer.train()

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir hf/runs

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

In [ ]:
predictions = trainer.predict(tokenized_dataset['test'])
predictions

In [ ]:
predicted_labels = np.argmax(predictions.predictions, axis=-1)
test_set = tokenized_dataset['test']
test_set = test_set.add_column('prediction_label', predicted_labels)
test_set = test_set.add_column('prediction', list(map(lambda label: id2category[label], predicted_labels)))
test_set

In [ ]:
columns = ['question', 'answer',  'prediction']
test_set.set_format('pandas', columns=columns)
df = test_set.to_pandas()[columns]
df.style.set_table_styles(
    [
        {'selector': 'td', 'props': [('word-wrap', 'break-word')]}
    ]
)
df.head(15)

In [ ]:
errors = df[df['answer'] != df['prediction']]
errors.head(15)

# 6. Definición del Positional Embedding

En esta sección se definen los embeddings utilizados por el modelo Transformer. Se generan las representaciones vectoriales de los tokens y se incorpora información sobre su posición en la secuencia mediante diferentes tipos de codificación posicional (sinusoidal o aprendible), produciendo las representaciones de entrada que serán utilizadas por el modelo.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)

for param in model.base_model.parameters():
    param.requires_grad = False


input_sizes = [inputs['input_ids'].shape] * 3
input_types = [inputs['input_ids'].dtype] * 3
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

In [ ]:
model.classifier

In [ ]:
import torch.nn as nn


classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(768, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 2)
)

# simplemente reemplazamos el clasificador existente por el nuestro:
model.classifier = classifier
with torch.no_grad():
    print(summary(model, input_size=input_sizes, dtypes=input_types, col_names=['input_size', 'output_size', 'num_params', 'trainable']))

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time

trainer.train()

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(category2id)).to(device)
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    processing_class=tokenizer
)

In [ ]:
%%time
trainer.train()

In [ ]:
model.eval()
trainer.evaluate(tokenized_dataset['test'])

In [ ]:
import numpy as np
import torch.nn as nn
from enum import Enum
from typing import Optional


class PosEncodingType(Enum):
    SINUSOID = 1
    LEARNABLE = 2


class SinusoidPE(nn.Module):

    def __init__(self, max_len: int, d_model: int):
        super(SinusoidPE, self).__init__()

        # Definimos un vector columna con las posiciones de la secuencia de entrada (pos)
        pos = torch.arange(max_len).unsqueeze(1)
        # Definimos un vector de fila con las dimensiones del embedding (i)
        i = torch.arange(d_model).unsqueeze(0)

        # Calculamos el denominador segun la formula
        div_term = 1 / torch.pow(10000, (2 * (i // 2)) / torch.tensor(d_model, dtype=torch.float32))
        # Aplicamos el denominador a las posiciones
        angle_rads = pos * div_term

        # Inicializamos la matriz de positional encodings
        pos_encoding = torch.zeros(max_len, d_model)
        # Calculamos los embeddings para los numeros pares con seno: PE(pos, 2i)
        pos_encoding[:, 0::2] = torch.sin(angle_rads[:, 0::2])
        # Calculamos los embdeddings para los numeros inpares con coseno: PE(pos, 2i+1)
        pos_encoding[:, 1::2] = torch.cos(angle_rads[:, 1::2])

        # Registramos la variable como atributo de clase
        self.register_buffer("pos_encoding", pos_encoding.unsqueeze(0), persistent=False)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos_encoding[:, :x.size(1), :]


class LearnablePE(nn.Module):

    def __init__(self, vocab_size: int, d_model: int, max_len: int = float('-inf')):
        super(LearnablePE, self).__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(0, max(x.size(-1), self.max_len))
        pos_emb = self.embedding(positions)
        return x + pos_emb



class TokenAndPosEmbedding(nn.Module):

    def __init__(self, max_len: int, embed_dim: int, vocab_size: int, pos_encoding_type: PosEncodingType = PosEncodingType.SINUSOID):
        super(TokenAndPosEmbedding, self).__init__()
        self.token_emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        if pos_encoding_type == PosEncodingType.SINUSOID:
            self.pos_emb = SinusoidPE(max_len, embed_dim)
        else:
            self.pos_emb = LearnablePE(vocab_size, embed_dim)


    def forward(self, x):
        token_emb = self.token_emb(x)
        return self.pos_emb(token_emb)

In [ ]:
emb_dim = 128 if not IN_COLAB else 256
tpe = TokenAndPosEmbedding(max_len, emb_dim, spanish_sentiment_tokenizer.vocab_size)
pos_encoding = tpe.pos_emb.pos_encoding.squeeze(0).numpy()

In [ ]:
text = "hola mundo!"
tokens = spanish_sentiment_tokenizer(text, max_length=max_len, truncation=True, padding='max_length')
x = torch.tensor(tokens['input_ids']).unsqueeze(0)
mask = torch.tensor(tokens['input_ids']).unsqueeze(0)
embedding = tpe(x)
embedding.shape

### Visualización del Positional Encoding

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap del encoding sinusoidal
im = axes[0].imshow(pos_encoding[:50, :64], aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_xlabel('Dimensión del embedding')
axes[0].set_ylabel('Posición en la secuencia')
axes[0].set_title('Positional Encoding Sinusoidal\n(primeras 50 posiciones, 64 dimensiones)')
plt.colorbar(im, ax=axes[0])

# Similitud coseno entre posiciones
sim = cosine_similarity(pos_encoding[:40])
im2 = axes[1].imshow(sim, cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('Similitud Coseno entre Posiciones\n(primeras 40 posiciones)')
axes[1].set_xlabel('Posición')
axes[1].set_ylabel('Posición')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

El heatmap muestra los patrones sinusoidales de la matriz de codificación posicional. La matriz de similitud coseno confirma que posiciones adyacentes en la secuencia tienen representaciones más similares entre sí, decayendo a medida que la distancia entre posiciones aumenta.

# 7. Definición del Multi-Head Attention

Este módulo permite que el modelo atienda simultáneamente a diferentes partes de la secuencia de entrada mediante múltiples cabezas de atención.

Este proceso se encarga de:

- Generar las representaciones Query, Key y Value a partir de las entradas.

- Calcular los pesos de atención mediante el producto punto escalado.

- Aplicar máscaras para ignorar tokens de relleno cuando sea necesario.

- Combinar las diferentes cabezas de atención para producir la representación final utilizada por el modelo.

In [ ]:
import math


class MultiHeadAttention(nn.Module):

    def __init__(self, embed_size: int, num_heads: int = 8):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.num_heads = num_heads
        assert embed_size & num_heads == 0, 'El tamaño del embedding debería ser divisible por el numero de cabezas'
        self.projection_dim = embed_size // num_heads
        self.query = nn.Linear(emb_dim, emb_dim)
        self.key = nn.Linear(emb_dim, emb_dim)
        self.value = nn.Linear(emb_dim, emb_dim)
        self.comibe_heads = nn.Linear(emb_dim, emb_dim)


    @staticmethod
    def _scaled_dot_product(q, k, v, mask=None):
        """scaled dot product.

        Esta función define el bloque mencionado.
        Aquí se hace la multiplicación de matrices
        entre los Q, K y V para luego calcular el
        score de atención.

        Nótese además que aquí aplicamos una máscara
        de atención. Esto se debe a que como estamos
        rellenando las cadenas cortas con un token que
        en si mismo no trae ningún significado, no queremos
        que la red desperdicie recursos operando sobre este
        token, entonces usamos la máscara para poner los valores
        de atención en numeros muy pequeños para que al
        calcular el score, estos no sobresalgan sobre los demás.
        """
        # d_k para el escalamiento
        d_k = q.size()[-1]

        # multiplicacion Q \cdot K^T
        attn_logits = torch.matmul(q, k.transpose(-2, -1))
        # escalamiento
        attn_logits = attn_logits / math.sqrt(d_k)

        # Se aplica la máscara
        if mask is not None:
            attn_logits = attn_logits.masked_fill(mask.reshape(mask.shape[0], 1, 1, -1) == 0, -9e-15)

        # Se calcula el score de atención.
        attention = torch.softmax(attn_logits, dim=-1)
        # Se obtienen los valores tras el score de atención.
        values = torch.matmul(attention, v)
        return values, attention


    def _separate_heads(self, x, batch_size):
        # Llega: (batch, seq_len, emb_dim)
        x =  x.reshape(batch_size, -1, self.num_heads, self.projection_dim)  # (batch, seq_len, num_heads, emb_dim / num_heads)
        return x.permute(0, 2, 1, 3)  # (batch, num_heads, seq_len, emb_dim / num_heads)


    def forward(self, x, mask=None, return_attention=False):
        """forward

        Este es todo el forward pass del multi-head attention.
        Aquí se coordina el resto de las operaciones, como
        la concatenación de las múltiples cabezas como
        el paso por la capa densa previo a entregar el
        resultado.
        """
        # x: (batch, seq_len, emb_dim)
        batch_size, seq_len, emb_dim = x.size()
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        q = self._separate_heads(q, batch_size)
        k = self._separate_heads(k, batch_size)
        v = self._separate_heads(v, batch_size)

        weights, attention = self._scaled_dot_product(q, k, v, mask)
        weights = weights.permute(0, 2, 1, 3) # (batch, seq_len, num_heads, emb_dim / num_heads)
        weights = weights.reshape(batch_size, seq_len, emb_dim)
        output = self.comibe_heads(weights)

        if return_attention:
            return output, attention
        else:
            return output

In [ ]:
mha = MultiHeadAttention(emb_dim)
mha(embedding, mask).shape

# 8. Definición del modulo Transformers

En esta sección se define un bloque Transformer que integra los componentes principales de esta arquitectura. El bloque aplica un mecanismo de multi-head attention para capturar relaciones entre los elementos de la secuencia, seguido de una red feed-forward que transforma las representaciones obtenidas. Además, se utilizan dropout y layer normalization para mejorar la estabilidad del entrenamiento y reducir el sobreajuste, produciendo así representaciones más robustas para el modelo.

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, emb_dim: int, num_heads: int = 8):
        super(TransformerBlock, self).__init__()
        self.mhatt = MultiHeadAttention(emb_dim, num_heads)
        self.mhatt_dropput = nn.Dropout(0.2)
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, emb_dim)
        )
        self.layer_norm1 = nn.LayerNorm(emb_dim)
        self.layer_norm2 = nn.LayerNorm(emb_dim)


    def forward(self, x, mask=None):
        attn_output = self.mhatt(x, mask)
        attn_output = self.mhatt_dropput(attn_output)
        attn_output = self.layer_norm1(attn_output)
        ffn_out = self.ffn(attn_output)
        return self.layer_norm2(ffn_out)

In [ ]:
tb = TransformerBlock(emb_dim)
tb(embedding, mask).shape

In [ ]:
num_heads = 8
vocab_size = spanish_sentiment_tokenizer.vocab_size

token_embeddings = TokenAndPosEmbedding(max_len, emb_dim, vocab_size)
transformer = TransformerBlock(emb_dim, num_heads)
ff = nn.Sequential(
    nn.Flatten(),
    nn.Linear(max_len * emb_dim, spanish_sentiment_dataset.num_classes)
)

In [ ]:
it = iter(train_loader)
batch = next(it)
x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']

embeddings = token_embeddings(x)
assert embeddings.shape == (train_loader.batch_size, max_len, emb_dim)

attention = transformer(embeddings, mask)
attention.shape

In [ ]:
pred = ff(attention)
pred.shape

# 9. Definición del clasificador

En esta sección se construye y entrena el modelo de clasificación de sentimientos basado en un Transformer. El proceso sigue los siguientes pasos:

- Se definen los embeddings de tokens y posiciones para representar el texto de entrada.

- Las representaciones pasan por un bloque Transformer con mecanismo de atención para capturar relaciones entre las palabras.

- La salida se envía a una red neuronal densa que actúa como clasificador para predecir el sentimiento.

- Se implementan los pasos de entrenamiento, validación y prueba, junto con el cálculo de métricas de desempeño.

- Finalmente, se configura el optimizador y el Trainer de PyTorch Lightning para entrenar el modelo.

Como se evidenció anteriormente, el dataset presenta un desbalance de clases. Para mitigar este problema, se implementará el uso de class weights, lo que permite penalizar en mayor medida los errores cometidos sobre la clase minoritaria durante el entrenamiento del modelo.

In [ ]:
from collections import Counter

labels = [x['label'] for x in spanish_sentiment_dataset.dataset]

counts = Counter(labels)

class_counts = [counts[i] for i in range(spanish_sentiment_dataset.num_classes)]

print("Distribución de clases:", class_counts)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy


class SentimentClassifier(LightningModule):

    def __init__(self, max_len, vocab_size, num_classes, emb_dim, class_counts, num_heads=8):
        super(SentimentClassifier, self).__init__()
        self.num_classes = num_classes

        counts = torch.tensor(class_counts, dtype=torch.float)
        total = counts.sum()
        weights = total / (num_classes * counts)
        self.register_buffer("class_weights", weights)

        self.token_embeddings = TokenAndPosEmbedding(max_len, emb_dim, vocab_size)
        self.transformer = TransformerBlock(emb_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(max_len * emb_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

        self.train_acc = Accuracy(task='binary', num_classes=num_classes)
        self.val_acc = Accuracy(task='binary', num_classes=num_classes)
        self.test_acc = Accuracy(task='binary', num_classes=num_classes)



    def forward(self, x, mask=None):
        embeddings = self.token_embeddings(x)
        attention = self.transformer(embeddings, mask)
        return self.classifier(attention)


    def training_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        loss = F.cross_entropy(y_hat, y, weight=self.class_weights)
        preds = torch.argmax(y_hat, dim=1)
        self.train_acc(preds, y)
        self.log('train-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train-acc', self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        loss = F.cross_entropy(y_hat, y, weight=self.class_weights)
        preds = torch.argmax(y_hat, dim=1)
        self.val_acc(preds, y)
        self.log('val-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val-acc', self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        preds = torch.argmax(y_hat, dim=1)
        self.test_acc(preds, y)
        self.log('test-acc', self.test_acc, prog_bar=True, on_step=False, on_epoch=True)


    def predict_step(self, batch):
        x, mask = batch['input_ids'], batch['attention_mask']
        return self(x, mask)


    def configure_optimizers(self):
        optimizer =  torch.optim.AdamW(self.parameters(), lr=2e-5, weight_decay=1e-5)
        return optimizer


model = SentimentClassifier(
    max_len=spanish_sentiment_dataset.seq_length,
    vocab_size=spanish_sentiment_tokenizer.vocab_size,
    num_classes=spanish_sentiment_dataset.num_classes,
    emb_dim=emb_dim,
    class_counts=class_counts
)

tb_logger = TensorBoardLogger('tb_logs', name='TransformersClassifier')
callbacks=[EarlyStopping(monitor='train-loss', patience=3, mode='min')]
trainer = Trainer(max_epochs=20, devices=1, logger=tb_logger, callbacks=callbacks, precision="16-mixed")

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir tb_logs/

In [ ]:
model.eval()
trainer.test(model, test_loader)

## Curvas de Entrenamiento

Visualizamos la evolución del loss y la accuracy durante el entrenamiento y la validación, leyendo directamente los logs guardados por TensorBoard. Esto permite identificar sobreajuste, convergencia temprana u otros comportamientos durante el entrenamiento.

In [ ]:
import glob
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

log_dir = 'tb_logs/TransformersClassifier'
versions = sorted(glob.glob(f'{log_dir}/version_*'))

if not versions:
    print('No se encontraron logs de entrenamiento en', log_dir)
else:
    ea = EventAccumulator(versions[-1])
    ea.Reload()
    tags = ea.Tags().get('scalars', [])
    print('Métricas disponibles:', tags)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for (train_tag, val_tag, ylabel), ax in zip(
        [('train-loss', 'val-loss', 'Loss'), ('train-acc', 'val-acc', 'Accuracy')],
        axes
    ):
        for tag, label, ls in [(train_tag, 'Train', '-'), (val_tag, 'Validación', '--')]:
            if tag in tags:
                events = ea.Scalars(tag)
                ax.plot([e.step for e in events], [e.value for e in events], linestyle=ls, label=label)
        ax.set_xlabel('Época')
        ax.set_ylabel(ylabel)
        ax.set_title(f'Curva de {ylabel}')
        ax.legend()

    plt.tight_layout()
    plt.show()

El modelo de clasificación de sentimientos basado en Transformer obtuvo una precisión del 85.29% en el conjunto de prueba, lo que significa que el modelo logra clasificar correctamente la mayoría de los comentarios como positivos o negativos. Este resultado muestra que el modelo aprendió a identificar patrones y relaciones entre las palabras de los textos para determinar el tipo de sentimiento expresado en cada comentario.

Aunque el desempeño es bueno, todavía existe un pequeño porcentaje de errores, por lo que el modelo podría mejorarse utilizando más datos de entrenamiento, ajustando algunos parámetros o probando versiones más avanzadas del modelo.

# 10. Hacemos predicciones

In [ ]:
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
predictions = torch.argmax(predictions, dim=-1)
predictions = [spanish_sentiment_dataset.id_2_class_map[pred] for pred in predictions.numpy()]

In [ ]:
import pandas as pd

# Obtener predicciones
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
pred_labels = torch.argmax(predictions, dim=-1).numpy()

# Obtener índices del test set
test_indices = test_dataset.indices

# Crear DataFrame con resultados
id_2_token = {v: k for k, v in vocab.items()}

df_results = pd.DataFrame({
    'texto': [dataset[i]['text'] for i in test_indices],
    'etiqueta_real': [dataset[i]['label'] for i in test_indices],
    'prediccion': pred_labels,
    'sentimiento_real': [sentiment_dataset.id_2_class_map[dataset[i]['label']] for i in test_indices],
    'sentimiento_pred': [sentiment_dataset.id_2_class_map[p] for p in pred_labels]
})

df_results['correcto'] = df_results['etiqueta_real'] == df_results['prediccion']
print(f"Precisión: {df_results['correcto'].mean():.4f}")
print(f"Total correctos: {df_results['correcto'].sum()} / {len(df_results)}")
df_results.head(10)

In [ ]:
# Análisis de errores
errors = df_results[~df_results['correcto']]
print(f"\nTotal de errores: {len(errors)}")
print(f"\nEjemplos de predicciones incorrectas:")
errors[['texto', 'sentimiento_real', 'sentimiento_pred']].head(10)

# 11. Analizamos los resultados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Matriz de confusión
cm = confusion_matrix(df_results['etiqueta_real'], df_results['prediccion'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Matriz de confusión
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j + 0.5, i + 0.5, str(cm[i, j]),
                     ha='center', va='center', color='black', fontsize=14, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=axes[0])

axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de Confusión')

# Plot 2: Distribución de predicciones
pred_counts = df_results['sentimiento_pred'].value_counts()
axes[1].bar(pred_counts.index, pred_counts.values, color=['salmon', 'lightgreen'])
axes[1].set_xlabel('Sentimiento')
axes[1].set_ylabel('Cantidad')
axes[1].set_title('Distribución de Predicciones')

plt.tight_layout()
plt.show()

# Reporte de clasificación
print("\nReporte de Clasificación:")
print(classification_report(df_results['etiqueta_real'], df_results['prediccion'],
                          target_names=['Negative', 'Positive']))

## Curva ROC y Precisión-Recall

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

model.eval()
all_probs, all_true = [], []

with torch.no_grad():
    for batch_data in test_loader:
        x_b    = batch_data['input_ids']
        mask_b = batch_data['attention_mask']
        y_b    = batch_data['y']
        logits = model(x_b, mask_b)
        probs  = torch.softmax(logits, dim=-1)[:, 1]   # probabilidad clase positiva
        all_probs.extend(probs.cpu().numpy())
        all_true.extend(y_b.cpu().numpy())

fpr, tpr, _        = roc_curve(all_true, all_probs)
roc_auc            = auc(fpr, tpr)
prec, rec, _       = precision_recall_curve(all_true, all_probs)
avg_precision      = average_precision_score(all_true, all_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Clasificador aleatorio')
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curva ROC')
axes[0].legend()

axes[1].plot(rec, prec, color='darkorange', lw=2, label=f'PR (AP = {avg_precision:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precisión-Recall')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"AUC-ROC: {roc_auc:.4f} | Average Precision: {avg_precision:.4f}")

Las curvas ROC y Precisión-Recall muestran que el modelo tiene una capacidad **moderada** para separar clases: su desempeño es claramente mejor que el azar, pero aún limitado para distinguir correctamente la clase minoritaria (negativa). Esto es consistente con el desbalance del dataset y con la distribución de predicciones observada, donde el modelo tiende a favorecer la clase positiva. En consecuencia, aunque la precisión global es aceptable, todavía hay margen de mejora en **recall y precisión de la clase negativa**.

## Análisis de Errores por Longitud de Texto

Analizamos si la longitud del texto (número de palabras) influye en el desempeño del modelo, identificando en qué rangos el clasificador comete más errores.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_results['longitud_palabras'] = df_results['texto'].apply(lambda t: len(t.split()))

bins       = [0, 5, 10, 20, 40, 200]
bin_labels = ['1-5', '6-10', '11-20', '21-40', '>40']
df_results['rango_longitud'] = pd.cut(
    df_results['longitud_palabras'], bins=bins, labels=bin_labels, right=True
)

acc_by_len = (
    df_results.groupby('rango_longitud', observed=True)['correcto']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'accuracy', 'count': 'n_muestras'})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(acc_by_len.index.astype(str), acc_by_len['accuracy'], color='steelblue')
axes[0].axhline(
    df_results['correcto'].mean(), color='red', linestyle='--',
    label=f"Media global ({df_results['correcto'].mean():.2f})"
)
axes[0].set_xlabel('Rango de longitud (palabras)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy por longitud de texto')
axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].bar(acc_by_len.index.astype(str), acc_by_len['n_muestras'], color='coral')
axes[1].set_xlabel('Rango de longitud (palabras)')
axes[1].set_ylabel('Número de muestras')
axes[1].set_title('Distribución de muestras por longitud')

plt.tight_layout()
plt.show()
print(acc_by_len.to_string())

Las gráficas sugieren que el rendimiento del modelo varía según la longitud del texto: en rangos con más ejemplos, la *precisión (accuracy)* se mantiene más estable y cercana al promedio global, mientras que en rangos con pocas muestras puede fluctuar más por efecto del tamaño de muestra. En general, no se observa una caída drástica del desempeño en textos largos, pero sí conviene interpretar con cuidado los extremos de longitud, donde hay menos datos y mayor incertidumbre.

## Inferencia en Texto Personalizado

Probamos el modelo con frases externas al dataset para evaluar su capacidad de generalización. Esto también sirve como demostración práctica del sistema completo de clasificación de sentimientos.

In [ ]:
import torch

def predecir_sentimiento(texto, seq_length=128):
    """Clasifica el sentimiento de un texto en español y devuelve las probabilidades."""
    model.eval()
    encoding = spanish_sentiment_tokenizer(
        texto,
        max_length=seq_length,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    with torch.no_grad():
        logits = model(encoding['input_ids'], encoding['attention_mask'])
        probs  = torch.softmax(logits, dim=-1)[0]
        pred   = torch.argmax(probs).item()
    return {
        'texto':         texto,
        'sentimiento':   sentiment_dataset.id_2_class_map[pred],
        'prob_positivo': probs[1].item(),
        'prob_negativo': probs[0].item(),
    }

textos_prueba = [
    "El hotel fue increíble, muy limpio y el personal muy amable.",
    "Pésimo servicio, la habitación estaba sucia y el ruido era insoportable.",
    "La ubicación estaba bien pero el desayuno dejaba mucho que desear.",
    "Volveré sin duda, una experiencia maravillosa de principio a fin.",
    "No recomiendo este lugar para nada, fue una decepción total.",
    "El precio era razonable pero el servicio podría mejorar.",
]

print(f"{'Texto':<62} {'Sentimiento':<12} {'P(pos)':<8} {'P(neg)'}")
print('-' * 96)
for texto in textos_prueba:
    r = predecir_sentimiento(texto)
    print(f"{r['texto'][:60]:<62} {r['sentimiento'].upper():<12} {r['prob_positivo']:.2%}    {r['prob_negativo']:.2%}")

Las 6 frases de prueba fueron clasificadas como **positive**, incluso aquellas con polaridad claramente negativa (por ejemplo: *“Pésimo servicio...”* y *“No recomiendo este lugar...”*).

- Hay **sesgo fuerte hacia la clase positiva**.
- Las probabilidades están en un rango estrecho (~0.66–0.69), lo que sugiere **baja capacidad de separación** entre clases en inferencia libre.
- Esto es consistente con lo observado antes: dataset desbalanceado y dificultad en clase negativa.

El modelo parece haber aprendido un patrón de decisión casi constante hacia “positivo”, por lo que su utilidad práctica para textos ambiguos/negativos es limitada sin ajustes adicionales.

# 12. Conclusiones

En este notebook se implementó y evaluó un modelo basado en arquitectura Transformer para la clasificación de sentimientos en español, para comparar el desempeño con un modelo LSTM bidireccional.

### Arquitectura

#### Modelo LSTM

- Word embeddings para representar las palabras.

- LSTM bidireccional para capturar dependencias en ambas direcciones del texto.

- Capas densas finales para realizar la clasificación del sentimiento.

#### Modelo Transformer

- Tokenización de los textos para convertirlos en representaciones numéricas.

- Embeddings de tokens y posiciones para representar el contenido y el orden de las palabras.

- Bloque de Multi-Head Attention para capturar relaciones entre palabras dentro de la secuencia.

- Capas densas finales para predecir el sentimiento.

### Resultados

- El modelo LSTM alcanzó una precisión aproximada del 87%, mostrando un buen desempeño general en la clasificación.

- El análisis del reporte de clasificación muestra que el modelo es mucho más preciso identificando comentarios positivos que negativos, lo cual está relacionado con el desbalance de clases presente en el dataset.

- El modelo Transformer obtuvo una precisión cercana al 83% haciendo tratamiento en el balance de clases en el conjunto de prueba, demostrando que también es capaz de aprender patrones relevantes en los textos mediante el mecanismo de atención.

### Comparación entre modelos

- El LSTM logró un desempeño ligeramente superior en precisión general, posiblemente debido a que el dataset es relativamente pequeño.

- El Transformer tiene la ventaja de capturar relaciones globales entre palabras mediante el mecanismo de atención, lo que lo hace especialmente útil en textos más largos o datasets más grandes.

- Ambos modelos muestran buen desempeño en la clase positiva, pero presentan mayores dificultades para clasificar correctamente comentarios negativos.

### Posibles mejoras

- Aumentar el tamaño del dataset para mejorar la capacidad de generalización del modelo.

- Usar modelos preentrenados como BERT o BETO, que suelen ofrecer mejores resultados en tareas de NLP.

- Ajustar hiperparámetros como tamaño de embeddings, número de capas o número de cabezas de atención.

También se evidenció que el uso de modelos basados en arquitecturas de tipo Transformer resulta más provechoso cuando se dispone de datasets de mayor tamaño. Esto se debe a que estos modelos cuentan con un número considerable de parámetros y están diseñados para capturar relaciones complejas y dependencias contextuales dentro del texto, lo cual requiere una cantidad suficiente de datos para ser aprendido adecuadamente.

En conclusión, ambos enfoques muestran buenos resultados para la clasificación de sentimientos en español. Sin embargo, los modelos basados en Transformers ofrecen mayor flexibilidad y escalabilidad, especialmente cuando se trabaja con grandes volúmenes de datos o modelos preentrenados.